# 💵 현금흐름 병목 & Lead-Lag Playground

`cash_flow_lead_lag_variable_guide.docx`의 **모든 파생변수**와 **추천 변수쌍**을 코드로 구현했습니다. 직접 변수를 골라 돌려보고 **해석을 덧붙이는** 용도입니다.

- **데이터:** `panel_long` (이제 `accounts_payable` 포함 → DIO/DSO/**DPO/CCC** 전부 계산 가능)
- **엔진:** `features.py`(파생변수 38종) · `cashflow_leadlag.py`(변수쌍 lead-lag) · `panel.py`(데이터) · `leadlag.py`(통계)
- **판정:** 상관 부호가 **기대부호와 일치 + p<0.05** → `가설지지`.

### 두 분석 축 (가이드 §4·§5)
1. **기업 내부** — 한 기업에서 운전자본 병목 `X_t` 가 현금흐름 `Y_{t+k}` 를 선행하는가
2. **기업 간** — 고객 `X` 가 공급사 `Y` 를 (또는 공급사→고객) 선행하는가

## Step 1 — 데이터 로드 & payables 확인

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import display
import panel as P, features as F, cashflow_leadlag as C
pd.set_option('display.max_rows', 200, 'display.width', 200)

long = P.load_long(P.DB_DEFAULT)
ap = long[long.item=='accounts_payable']
print('panel_long:', long.shape, '| accounts_payable rows:', len(ap),
      '| payables 보유 기업:', ap.ticker.nunique())

## Step 2 — 파생변수 카탈로그 (쓸 수 있는 변수 전체)

문서 §3의 변수를 모두 구현했습니다. `feature` 열의 이름을 그대로 아래 스텝에 넣으면 됩니다.

In [ ]:
cat = F.feature_catalog()
for kind in ['amount','days','growth','spread','ratio']:
    print(f'── {kind} ──')
    display(cat[cat.kind==kind][['feature','default_transform','설명']].reset_index(drop=True))

## Step 3 — 한 기업의 feature 패널 보기

원천+파생 변수가 분기별로 어떻게 계산됐는지 확인합니다. `TICKER`를 바꿔보세요.

In [ ]:
TICKER = 'MU'
f = F.compute_features(long, TICKER)
print(f'{P.company_name(long,TICKER)} feature 패널:', f.shape)
display(f[['DIO','DSO','DPO','CCC','OCF_MARGIN','FCF_MARGIN','CAPEX_TO_OCF']].dropna(how='all').tail(6).round(2))

## Step 4 — 기업 내부: 한 변수쌍 상세 (핵심)

`X_t → Y_{t+k}` 를 lag 0~4로 스캔합니다. `plot_pair`가 시계열 + 시차상관을 그려줍니다. 기대부호(`EXPECTED`)와 대조해 판정합니다.

In [ ]:
# ============== EDIT ==============
TICKER   = 'MU'
X_FEAT, X_TF = 'CCC', 'level'          # 선행 후보 (병목)
Y_FEAT, Y_TF = 'FCF_MARGIN', 'level'   # 후행 (현금 결과)
EXPECTED = '-'                          # 기대부호 '+'/'-'
# =================================

res = C.analyze_internal(long, TICKER, X_FEAT, X_TF, Y_FEAT, Y_TF, EXPECTED)
display(pd.Series({k:v for k,v in res.items() if k!='_scan'}))
print('전체 시차 상관표:'); display(res['_scan']['table'].round(3))
C.plot_pair(long, 'internal', TICKER, None, X_FEAT, X_TF, Y_FEAT, Y_TF, EXPECTED);

## Step 5 — 기업 내부: 추천 Top-8 × 여러 기업

가이드 §4의 Top-8 변수쌍을 지정한 기업들에 대해 한 번에 돌립니다.

In [ ]:
TICKERS = ['NVDA','AMD','MU','INTC','AMAT','LRCX','AMKR','DELL','AVGO','STX']
internal = C.run_internal(long, TICKERS)
print('가설지지:', ((internal.status=='ok') & internal.sign_match & internal.significant).sum(), '건')
# 가설지지된 것만 보기
hit = internal[(internal.sign_match) & (internal.significant)]
display(hit[['company','X','Y','expected','best_lag','pearson','p_value','granger_p','verdict']])

## Step 6 — 기업 간: 한 변수쌍 상세

고객 섹션 X → 공급사 섹션 Y. `DIRECTION='c2s'`(고객→공급사) 또는 `'s2c'`(공급사→고객).

In [ ]:
# ============== EDIT ==============
CUSTOMER = 'hyperscalers'
SUPPLIER = 'ai_chip'
DIRECTION = 'c2s'
X_FEAT, X_TF = 'DPO', 'level'          # 선행 (고객)
Y_FEAT, Y_TF = 'DSO', 'level'          # 후행 (공급사)
EXPECTED = '+'; LAGS = (0,3)
# =================================

res = C.analyze_cross(long, CUSTOMER, SUPPLIER, DIRECTION, X_FEAT, X_TF, Y_FEAT, Y_TF, EXPECTED, LAGS)
display(pd.Series({k:v for k,v in res.items() if k!='_scan'}))
C.plot_pair(long, 'cross', CUSTOMER, SUPPLIER, X_FEAT, X_TF, Y_FEAT, Y_TF, EXPECTED, LAGS, DIRECTION);

## Step 7 — 기업 간: 추천 변수쌍 × 전체 관계

가이드 §5의 7개 변수쌍을 밸류체인 관계 전체에 적용합니다.

In [ ]:
cross = C.run_cross(long)
ok = cross[cross.status=='ok']
print(f'분석가능 {len(ok)} / 전체 {len(cross)} · 가설지지 {((ok.sign_match)&(ok.significant)).sum()}건')
hit = ok[(ok.sign_match) & (ok.significant)].sort_values('p_value')
display(hit[['customer','supplier','X','Y','best_lag','pearson','p_value','granger_p','verdict']])

## Step 8 — 결과 필터링 (해석 후보 추리기)

가이드 §10의 순서대로: ①기대부호 일치 ②유의 ③시차 안정성·표본수 확인. 아래에서 원하는 조건으로 필터해 해석 후보를 좁히세요.

In [ ]:
# 예: 표본 12개 이상 + 가설지지 + Granger도 유의한 강한 후보만
strong = cross[(cross.status=='ok') & cross.sign_match & cross.significant &
               (cross.n>=12) & (cross.granger_p.fillna(1)<0.1)]
display(strong[['customer','supplier','X','Y','best_lag','pearson','p_value','granger_p','hypothesis']])

## Step 9 — 리포트(MD) 생성

내부/기업간 결과표 + 직접 해석을 적을 여백이 포함된 마크다운을 저장합니다.

In [ ]:
import os
os.makedirs('reports', exist_ok=True)
path = C.generate_md(internal, cross, 'reports/cashflow_leadlag_2026-07-06.md')
print('저장:', path)